In [1]:
# Import các thư viện cần thiết
import pandas as pd
import numpy as np
import re
import ast
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity, linear_kernel
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Load dataset và xem thông tin tổng quan
df = pd.read_csv('../data/all_recipes_final.csv')
print(f"Dataset shape: {df.shape}")
print(f"Phân bố theo nguồn:")
print(df['source'].value_counts())
df.head()


Dataset shape: (10263, 12)
Phân bố theo nguồn:
source
dienmayxanh    8993
vnexpress       737
vncooking       533
Name: count, dtype: int64


,title,type_of_food,link,description,ingredients,step,note,num_of_ingredients,cook_time,num_of_people,calories,source
0,Cách muối dưa hành truyền thống,Món Tết,https://vnexpress.net/doi-song-cooking-cach-mu...,Dưa hành muối là món ăn truyền thống ngày Tết ...,"['1 kg hành củ tươi', 'Tro bếp hoặc nước vo gọ...",['Bước 1: Chọn hành củ: Nên chọn hành củ ta bá...,[],5,45 phút,8-10 người,459 kcal,vnexpress
1,Su hào xào mực - món cổ Tết Bát Tràng,Món Tết,https://vnexpress.net/doi-song-cooking-su-hao-...,Đĩa xào khô ráo với su hào giòn ngọt quyện với...,"['2 củ su hào non', '1 con mực khô', '1/2 củ c...",['Bước 1: Chọn và sơ chế mực: Người dân làng g...,['Su hào xào mực cùng với canh măng mực là hai...,6,50 phút,4 - 5 người,1.162 kcal,vnexpress
2,Canh măng ngày Tết cổ truyền Hà Nội,Món Tết,https://vnexpress.net/doi-song-cooking-canh-ma...,"Măng ngấu vị, giòn ngon, móng giò hầm vừa độ s...","['800 gr măng khô', '2 móng giò lợn', 'Nước dù...","['Bước 1: Chọn măng khô: Theo lối cũ, người nộ...",['Nếu tận dụng nước luộc gà nấu canh măng thì ...,6,100 phút,8 - 10 người,4.930 kcal,vnexpress
3,Giả hạnh nhân - món ngon Tết xưa Hà Nội,Món Tết,https://vnexpress.net/doi-song-cooking-gia-han...,Đây là món ăn cổ truyền thường thấy trong cỗ T...,"['2 bộ lòng mề gà', '100 gr lạc', '50 gr hạt đ...",['Bước 1: Chọn và sơ chế lạc: Chọn lạc khô chắ...,['Hạnh nhân xào (hay giả hạnh nhân) là món ăn ...,8,60 phút,4-5 người,1.112 kcal,vnexpress
4,Chả bì ớt xiêm xanh,Món Tết,https://vnexpress.net/doi-song-cooking-cha-bi-...,"Chả bì bóng đẹp, gói đều tay. Khi ăn vị ngọt m...","['500 gr giò sống', '300 gr bì lợn', '20 - 30 ...","['Bước 1: Chọn và sơ chế bì lợn, chuẩn bị giò ...",['Nên sơ chế kỹ bì lợn để chả được thơm. Tùy t...,6,60 phút,5-6 người,2.512 kcal,vnexpress


## 1. Data Preprocessing

In [3]:
# Xử lý missing values
data = df.copy()

data['title'] = data['title'].fillna('')
data['description'] = data['description'].fillna('')
data['step'] = data['step'].fillna('[]')
data['ingredients'] = data['ingredients'].fillna('[]')
data['type_of_food'] = data['type_of_food'].fillna('Unknown')

print("Missing values after handling:")
print(data[['title', 'description', 'step', 'ingredients', 'type_of_food', 'calories', 'cook_time']].isnull().sum())


Missing values after handling:
title              0
description        0
step               0
ingredients        0
type_of_food       0
calories        9826
cook_time        295
dtype: int64


In [4]:
# Định nghĩa các hàm xử lý và làm sạch dữ liệu
def parse_list_string(s):
    if pd.isna(s) or s == '[]':
        return []
    try:
        return ast.literal_eval(s)
    except:
        return []

def clean_text(text):
    if pd.isna(text):
        return ''
    text = str(text).lower()
    text = re.sub(r'[^\w\s\u00C0-\u1EF9]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def parse_cook_time(time_str):
    if pd.isna(time_str):
        return np.nan
    time_str = str(time_str).lower()
    minutes = 0
    
    hour_match = re.search(r'(\d+)\s*(?:giờ|h|hour)', time_str)
    if hour_match:
        minutes += int(hour_match.group(1)) * 60
    
    min_match = re.search(r'(\d+)\s*(?:phút|p|min|minute)', time_str)
    if min_match:
        minutes += int(min_match.group(1))
    
    if minutes == 0:
        num_match = re.search(r'(\d+)', time_str)
        if num_match:
            minutes = int(num_match.group(1))
    
    return minutes if minutes > 0 else np.nan

def parse_calories(cal_str):
    if pd.isna(cal_str):
        return np.nan
    cal_str = str(cal_str).replace('.', '').replace(',', '')
    match = re.search(r'(\d+)', cal_str)
    if match:
        return float(match.group(1))
    return np.nan


In [5]:
# Áp dụng các hàm preprocessing lên dữ liệu
data['ingredients_list'] = data['ingredients'].apply(parse_list_string)
data['step_list'] = data['step'].apply(parse_list_string)

data['title_clean'] = data['title'].apply(clean_text)
data['description_clean'] = data['description'].apply(clean_text)
data['step_clean'] = data['step_list'].apply(lambda x: ' '.join([clean_text(s) for s in x]))

data['cook_time_minutes'] = data['cook_time'].apply(parse_cook_time)
data['calories_numeric'] = data['calories'].apply(parse_calories)

data['ingredients_clean'] = data['ingredients_list'].apply(
    lambda x: set([clean_text(ing) for ing in x if ing])
)

data[['title', 'title_clean', 'cook_time', 'cook_time_minutes', 'calories', 'calories_numeric']].head()


,title,title_clean,cook_time,cook_time_minutes,calories,calories_numeric
0,Cách muối dưa hành truyền thống,cách muối dưa hành truyền thống,45 phút,45.0,459 kcal,459.0
1,Su hào xào mực - món cổ Tết Bát Tràng,su hào xào mực món cổ tết bát tràng,50 phút,50.0,1.162 kcal,1162.0
2,Canh măng ngày Tết cổ truyền Hà Nội,canh măng ngày tết cổ truyền hà nội,100 phút,100.0,4.930 kcal,4930.0
3,Giả hạnh nhân - món ngon Tết xưa Hà Nội,giả hạnh nhân món ngon tết xưa hà nội,60 phút,60.0,1.112 kcal,1112.0
4,Chả bì ớt xiêm xanh,chả bì ớt xiêm xanh,60 phút,60.0,2.512 kcal,2512.0


In [6]:
# Kiểm tra kết quả preprocessing
print("Sample ingredients_clean:")
for i, ing in enumerate(data['ingredients_clean'].head(3)):
    print(f"\nRecipe {i+1}: {data['title'].iloc[i]}")
    print(f"Ingredients: {ing}")

Sample ingredients_clean:

Recipe 1: Cách muối dưa hành truyền thống
Ingredients: {'tro bếp hoặc nước vo gọa', 'lọ sạch', 'cà rốt trang trí tùy chọn', '1 kg hành củ tươi', 'muối hạt đường'}

Recipe 2: Su hào xào mực - món cổ Tết Bát Tràng
Ingredients: {'gia vị mắm muối đường hạt tiêu rượu trắng gừng', '1 con mực khô', '2 củ su hào non', 'rau mùi trang trí', 'mỡ lợn hoặc dầu ăn', '1 2 củ cà rốt'}

Recipe 3: Canh măng ngày Tết cổ truyền Hà Nội
Ingredients: {'800 gr măng khô', 'nước vo gạo ngâm măng', 'nước dùng gà hoặc ninh xương lợn', '2 móng giò lợn', 'hành khô hành củ', 'gia vị nước mắm truyền thống muối'}


## 2. TF-IDF Based Recommendation

Tính toán similarity dựa trên nội dung văn bản (title, description, steps) sử dụng TF-IDF và cosine similarity.


In [7]:
# Tạo TF-IDF matrix từ text features
data['combined_text'] = data['title_clean'] + ' ' + data['description_clean'] + ' ' + data['step_clean']

tfidf_vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=1,
    max_df=0.95
)

tfidf_matrix = tfidf_vectorizer.fit_transform(data['combined_text'])
print(f"TF-IDF Matrix shape: {tfidf_matrix.shape}")


TF-IDF Matrix shape: (10263, 5000)


In [8]:
# Tính cosine similarity matrix từ TF-IDF
tfidf_similarity = cosine_similarity(tfidf_matrix, tfidf_matrix)
print(f"TF-IDF Similarity Matrix shape: {tfidf_similarity.shape}")


TF-IDF Similarity Matrix shape: (10263, 10263)


In [9]:
# Hàm lấy recommendations dựa trên TF-IDF
def get_tfidf_recommendations(recipe_idx, similarity_matrix, df, top_n=5):
    sim_scores = list(enumerate(similarity_matrix[recipe_idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:top_n+1]
    
    recipe_indices = [i[0] for i in sim_scores]
    scores = [i[1] for i in sim_scores]
    
    result = df.iloc[recipe_indices][['title', 'type_of_food', 'calories', 'cook_time']].copy()
    result['tfidf_score'] = scores
    
    return result

def recommend_by_title_tfidf(title, df, similarity_matrix, top_n=5):
    matches = df[df['title'].str.contains(title, case=False, na=False)]
    if len(matches) == 0:
        print(f"No recipe found with title containing: {title}")
        return None
    
    recipe_idx = matches.index[0]
    print(f"\nInput: {df.loc[recipe_idx, 'title']}")
    print(f"   Type: {df.loc[recipe_idx, 'type_of_food']}, Calories: {df.loc[recipe_idx, 'calories']}")
    print("\nTF-IDF Recommendations:")
    
    return get_tfidf_recommendations(recipe_idx, similarity_matrix, df, top_n)


## 4. Ingredient TF-IDF Based Recommendation

**Ingredient TF-IDF** xử lý ingredient list như text documents, tốt vì:
- Xử lý được variations trong cách viết (thịt bò, bò, beef...)
- Gán trọng số cho ingredients based on importance
- Không bị ảnh hưởng bởi exact string matching

**Ý tưởng**: Mỗi recipe là 1 document, ingredients là words. Áp dụng TF-IDF để tính similarity.

In [10]:
# Chuẩn bị ingredient text cho TF-IDF
# Join ingredients thành string separated by spaces
data['ingredients_text'] = data['ingredients_list'].apply(
    lambda x: ' '.join([clean_text(ing) for ing in x if ing])
)

print("Sample ingredient texts:")
for i in range(3):
    print(f"\n{data['title'].iloc[i]}")
    print(f"   Ingredients text: {data['ingredients_text'].iloc[i][:100]}...")

# Build TF-IDF vectorizer cho ingredients
print("\nBuilding Ingredient TF-IDF matrix...")
ingredient_tfidf_vectorizer = TfidfVectorizer(
    max_features=2000,  # Fewer features than text-based
    ngram_range=(1, 2),  # Unigrams and bigrams
    min_df=2,
    max_df=0.8
)

ingredient_tfidf_matrix = ingredient_tfidf_vectorizer.fit_transform(data['ingredients_text'])
print(f"Ingredient TF-IDF Matrix shape: {ingredient_tfidf_matrix.shape}")

# Tính cosine similarity
ingredient_tfidf_similarity = cosine_similarity(ingredient_tfidf_matrix, ingredient_tfidf_matrix)
print(f"Ingredient TF-IDF Similarity Matrix shape: {ingredient_tfidf_similarity.shape}")

Sample ingredient texts:

Cách muối dưa hành truyền thống
   Ingredients text: 1 kg hành củ tươi tro bếp hoặc nước vo gọa muối hạt đường cà rốt trang trí tùy chọn lọ sạch...

Su hào xào mực - món cổ Tết Bát Tràng
   Ingredients text: 2 củ su hào non 1 con mực khô 1 2 củ cà rốt gia vị mắm muối đường hạt tiêu rượu trắng gừng rau mùi t...

Canh măng ngày Tết cổ truyền Hà Nội
   Ingredients text: 800 gr măng khô 2 móng giò lợn nước dùng gà hoặc ninh xương lợn hành khô hành củ gia vị nước mắm tru...

Building Ingredient TF-IDF matrix...
Ingredient TF-IDF Matrix shape: (10263, 2000)
Ingredient TF-IDF Similarity Matrix shape: (10263, 10263)


In [11]:
# Hàm lấy recommendations dựa trên Ingredient TF-IDF
def get_ingredient_tfidf_recommendations(recipe_idx, similarity_matrix, df, top_n=5):
    sim_scores = list(enumerate(similarity_matrix[recipe_idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:top_n+1]
    
    recipe_indices = [i[0] for i in sim_scores]
    scores = [i[1] for i in sim_scores]
    
    result = df.iloc[recipe_indices][['title', 'type_of_food', 'calories', 'cook_time']].copy()
    result['ing_tfidf_score'] = scores
    
    return result

def recommend_by_title_ingredient_tfidf(title, df, similarity_matrix, top_n=5):
    matches = df[df['title'].str.contains(title, case=False, na=False)]
    if len(matches) == 0:
        print(f"No recipe found with title containing: {title}")
        return None
    
    recipe_idx = matches.index[0]
    print(f"\nInput: {df.loc[recipe_idx, 'title']}")
    print(f"   Ingredients: {df.loc[recipe_idx, 'ingredients_text'][:100]}...")
    print("\nIngredient TF-IDF Recommendations:")
    
    return get_ingredient_tfidf_recommendations(recipe_idx, similarity_matrix, df, top_n)

print("Ingredient TF-IDF recommendation functions ready!")

Ingredient TF-IDF recommendation functions ready!


In [12]:
# Test Ingredient TF-IDF Recommendation
test_recipe = "Thịt bò xào"
display(recommend_by_title_ingredient_tfidf(test_recipe, data, ingredient_tfidf_similarity, top_n=10))


Input: Thịt bò xào hoa thiên lý nhanh gọn, bổ dưỡng cho ngày hè
   Ingredients: 300 gr hoa thiên lý 200 gr thịt bò 3 4 tép tỏi ớt tùy chọn gia vị mắm dầu hào hạt nêm hạt tiêu dầu ă...

Ingredient TF-IDF Recommendations:


,title,type_of_food,calories,cook_time,ing_tfidf_score
111,Bò thuôn hành răm món 'quốc dân' đậm vị Hà Nội,Món ngon hàng ngày,820 kcal,20 phút,0.540983
107,Niễng xào thịt bò - đặc sản ''trời ban'' vào đông,Món ngon hàng ngày,735 kcal,25 phút,0.540284
100,Tép đồng rang ba chỉ,Món ngon hàng ngày,871 kcal,30 phút,0.516596
708,"Canh ghẹ nấu rau muống đơn giản, thanh mát",Thực đơn cho ngày nắng nóng,818 kcal,30 phút,0.451876
168,"Tôm rang thịt ba chỉ măn ngọt, béo ngậy ngon cơm",Món ngon hàng ngày,1.058 kcal,35 phút,0.446186
124,Ếch xào măng - món chân quê vào nhà hàng,Món ngon hàng ngày,940 kcal,35 phút,0.430438
116,Bò xào sả ớt mềm ngon chỉ trong 15 phút,Món ngon hàng ngày,860 kcal,15 phút,0.421084
701,Bầu xào trứng - món đơn giản mà đưa cơm ngày hè,Thực đơn cho ngày nắng nóng,643 kcal,15 phút,0.419532
109,Tim heo xào chua ngọt,Món ngon hàng ngày,883 kcal,35 phút,0.408745
359,Rạm kho lá lốt – món ngon Thái Bình,Món ngon hàng ngày,708 kcal,40 phút,0.400685


## 3. Hybrid Approach

Kết hợp 2 phương pháp với trọng số: TF-IDF (0.4) + Ingredient TF-IDF (0.6)

**Rationale**: Ingredients quan trọng nhất trong food recommendation, nên tăng Ingredient TF-IDF weight

In [13]:
# Hàm tính hybrid similarity (kết hợp TF-IDF và Ingredient TF-IDF)
def compute_hybrid_similarity(tfidf_sim, ing_tfidf_sim, 
                              w_tfidf=0.4, w_ing_tfidf=0.6):
    total_weight = w_tfidf + w_ing_tfidf
    w_tfidf /= total_weight
    w_ing_tfidf /= total_weight
    
    print(f"Weights: TF-IDF={w_tfidf:.2f}, Ingredient TF-IDF={w_ing_tfidf:.2f}")
    
    hybrid_similarity = (w_tfidf * tfidf_sim + 
                        w_ing_tfidf * ing_tfidf_sim)
    
    return hybrid_similarity

In [14]:
# Tính hybrid similarity matrix
hybrid_similarity_matrix = compute_hybrid_similarity(
    tfidf_similarity, 
    ingredient_tfidf_similarity,
    w_tfidf=0.4,
    w_ing_tfidf=0.6
)
print(f"Hybrid Similarity Matrix shape: {hybrid_similarity_matrix.shape}")

Weights: TF-IDF=0.40, Ingredient TF-IDF=0.60
Hybrid Similarity Matrix shape: (10263, 10263)


In [15]:
# Hàm lấy recommendations dựa trên hybrid approach
def get_hybrid_recommendations(recipe_idx, hybrid_sim, tfidf_sim, ing_tfidf_sim, df, top_n=5):
    sim_scores = list(enumerate(hybrid_sim[recipe_idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:top_n+1]
    
    recipe_indices = [i[0] for i in sim_scores]
    hybrid_scores = [i[1] for i in sim_scores]
    
    result = df.iloc[recipe_indices][['title', 'type_of_food', 'calories', 'cook_time']].copy()
    result['hybrid_score'] = hybrid_scores
    result['tfidf_score'] = [tfidf_sim[recipe_idx][i] for i in recipe_indices]
    result['ing_tfidf_score'] = [ing_tfidf_sim[recipe_idx][i] for i in recipe_indices]
    
    return result

def recommend_by_title_hybrid(title, df, hybrid_sim, tfidf_sim, ing_tfidf_sim, top_n=5):
    matches = df[df['title'].str.contains(title, case=False, na=False)]
    if len(matches) == 0:
        print(f"❌ No recipe found with title containing: {title}")
        return None
    
    recipe_idx = matches.index[0]
    print(f"\nInput: {df.loc[recipe_idx, 'title']}")
    print(f"   Type: {df.loc[recipe_idx, 'type_of_food']}, Calories: {df.loc[recipe_idx, 'calories']}, Time: {df.loc[recipe_idx, 'cook_time']}")
    print("\nHybrid Recommendations:")
    
    return get_hybrid_recommendations(recipe_idx, hybrid_sim, tfidf_sim, ing_tfidf_sim, df, top_n)

## 5. Keyword-Based Recommendation

Extract keywords chính từ title (nguyên liệu chính + phương pháp nấu).

**Ưu điểm**: Đơn giản, focus vào main keywords, dễ giải thích

In [16]:
# Hàm lấy recommendations dựa trên keywords
def get_keyword_recommendations(recipe_idx, similarity_matrix, df, top_n=5):
    sim_scores = list(enumerate(similarity_matrix[recipe_idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:top_n+1]
    
    recipe_indices = [i[0] for i in sim_scores]
    scores = [i[1] for i in sim_scores]
    
    result = df.iloc[recipe_indices][['title', 'type_of_food', 'calories', 'cook_time']].copy()
    result['keyword_score'] = scores
    
    return result

def recommend_by_title_keyword(title, df, similarity_matrix, top_n=5):
    matches = df[df['title'].str.contains(title, case=False, na=False)]
    if len(matches) == 0:
        print(f"No recipe found with title containing: {title}")
        return None
    
    recipe_idx = matches.index[0]
    print(f"\nInput: {df.loc[recipe_idx, 'title']}")
    print(f"   Keywords: {df.loc[recipe_idx, 'keywords']}")
    print("\nKeyword-Based Recommendations:")
    
    return get_keyword_recommendations(recipe_idx, similarity_matrix, df, top_n)

In [17]:
# Hàm extract keywords/tags từ title
def extract_keywords(title):
    """
    Extract main keywords from recipe title
    Focus on: ingredients, cooking methods, dish types
    """
    if pd.isna(title):
        return set()
    
    title = clean_text(title)
    
    # Common Vietnamese cooking methods and dish types
    cooking_methods = ['xào', 'nướng', 'luộc', 'chiên', 'hấp', 'kho', 'rim', 'rang', 
                       'canh', 'súp', 'cháo', 'gỏi', 'nộm', 'salad', 'bún', 'phở', 
                       'mì', 'cơm', 'bánh', 'chè', 'sinh tố']
    
    # Common ingredients
    main_ingredients = ['thịt', 'bò', 'gà', 'heo', 'lợn', 'cá', 'tôm', 'mực', 'nghêu',
                        'rau', 'củ', 'quả', 'trứng', 'đậu', 'nấm', 'măng', 'bí', 
                        'cà', 'khoai', 'su', 'hào', 'cải', 'rau muống', 'rau cần']
    
    # Extract keywords
    keywords = set()
    words = title.split()
    
    # Add cooking methods
    for method in cooking_methods:
        if method in title:
            keywords.add(method)
    
    # Add ingredients (check for 2-word and 1-word matches)
    for i in range(len(words)):
        # Check 2-word combinations
        if i < len(words) - 1:
            two_word = f"{words[i]} {words[i+1]}"
            for ing in main_ingredients:
                if ing in two_word:
                    keywords.add(ing)
        
        # Check single words
        for ing in main_ingredients:
            if ing in words[i]:
                keywords.add(ing)
    
    # Add all significant words (length > 2) as backup
    for word in words:
        if len(word) > 2:
            keywords.add(word)
    
    return keywords

# Apply keyword extraction to all recipes
data['keywords'] = data['title'].apply(extract_keywords)
print(f"Keyword extraction completed!")

# Show examples
print("\nSample keywords:")
for i in range(5):
    print(f"\n{data['title'].iloc[i]}")
    print(f"   Keywords: {data['keywords'].iloc[i]}")

Keyword extraction completed!

Sample keywords:

Cách muối dưa hành truyền thống
   Keywords: {'truyền', 'cá', 'cách', 'muối', 'dưa', 'hành', 'thống'}

Su hào xào mực - món cổ Tết Bát Tràng
   Keywords: {'hào', 'bát', 'tết', 'su', 'xào', 'mực', 'tràng', 'món'}

Canh măng ngày Tết cổ truyền Hà Nội
   Keywords: {'truyền', 'măng', 'tết', 'nội', 'canh', 'ngày', 'gà'}

Giả hạnh nhân - món ngon Tết xưa Hà Nội
   Keywords: {'nhân', 'ngon', 'tết', 'giả', 'nội', 'hạnh', 'món', 'xưa'}

Chả bì ớt xiêm xanh
   Keywords: {'xanh', 'xiêm', 'chả'}


In [18]:
# Helper functions cho Keyword similarity (sử dụng Jaccard cho keyword sets)
def jaccard_similarity(set1, set2):
    """
    Tính Jaccard similarity giữa 2 sets
    J(A,B) = |A ∩ B| / |A ∪ B|
    """
    if len(set1) == 0 and len(set2) == 0:
        return 0.0
    intersection = len(set1.intersection(set2))
    union = len(set1.union(set2))
    return intersection / union if union > 0 else 0.0

def compute_jaccard_similarity_matrix(items_list):
    """
    Tính Jaccard similarity matrix cho list of sets
    Sử dụng cho keyword-based recommendation
    """
    n = len(items_list)
    similarity_matrix = np.zeros((n, n))
    
    for i in range(n):
        for j in range(i, n):
            sim = jaccard_similarity(items_list[i], items_list[j])
            similarity_matrix[i][j] = sim
            similarity_matrix[j][i] = sim
    
    return similarity_matrix

print("Jaccard helper functions defined (for keyword similarity only)!")

Jaccard helper functions defined (for keyword similarity only)!


In [19]:
# Tính Keyword-based similarity matrix
keywords_sets = data['keywords'].tolist()
keyword_similarity_matrix = compute_jaccard_similarity_matrix(keywords_sets)
print(f"Keyword Similarity Matrix shape: {keyword_similarity_matrix.shape}")

Keyword Similarity Matrix shape: (10263, 10263)


In [20]:
# So sánh Ingredient TF-IDF vs Keyword
test_recipe = "canh chua"

print("="*100)
print("INGREDIENT TF-IDF (Better Ingredient Matching)")
print("="*100)
display(recommend_by_title_ingredient_tfidf(test_recipe, data, ingredient_tfidf_similarity, top_n=5))

print("\n" + "="*100)
print("KEYWORD-BASED")
print("="*100)
display(recommend_by_title_keyword(test_recipe, data, keyword_similarity_matrix, top_n=5))

INGREDIENT TF-IDF (Better Ingredient Matching)

Input: Canh chua cá khoai mềm, ngọt tự nhiên
   Ingredients: 400 gr cá khoai 3 quả cà chua 1 2 quả dứa 1 2 quả chanh 3 củ hành khô 1 nhánh gừng nhỏ gia vị mắm mu...

Ingredient TF-IDF Recommendations:


,title,type_of_food,calories,cook_time,ing_tfidf_score
104,Nấu canh cá dọc mùng ấm ngày se lạnh,Món ngon hàng ngày,1.170 kcal,45 phút,0.549792
449,Cá thu sốt cà chua - món ăn quốc dân cho ngày ...,Món ngon ngày lạnh,1.174 kcal,40 phút,0.526179
458,"Bắp cải cuốn thịt mềm ngon, đậm vị",Món ngon ngày lạnh,1.116 kcal,50 phút,0.520746
455,Canh cá nấu su hào ngọt ấm ngày lạnh,Món ngon ngày lạnh,1.169 kcal,45 phút,0.507934
393,Cách nầu canh moi nấu dưa chua - món đưa cơm n...,Món ngon hàng ngày,478 kcal,30 phút,0.500426



KEYWORD-BASED

Input: Canh chua cá khoai mềm, ngọt tự nhiên
   Keywords: {'mềm', 'nhiên', 'khoai', 'cá', 'ngọt', 'chua', 'kho', 'canh'}

Keyword-Based Recommendations:


,title,type_of_food,calories,cook_time,keyword_score
235,Canh chua cá trê,Món ngon hàng ngày,NaN,NaN,0.333333
583,Canh chua cá Nam bộ,Món ngon theo vùng miền,NaN,NaN,0.333333
940,Canh chua cá lóc,Món chính,NaN,30phút,0.333333
1597,"Canh cá khoai rau cải ngọt ngon, dinh dưỡng ch...",Món canh,NaN,15 phút,0.333333
1440,Canh cá khoai cải cúc (tần ô) ngọt mát đưa cơm...,Món canh,NaN,15 phút,0.312500


In [21]:
# Class FoodRecommender hoàn chỉnh - với Ingredient TF-IDF và Keyword-based
class FoodRecommender:
    
    def __init__(self, df):
        self.df = df.copy()
        self._preprocess()
        self._build_similarity_matrices()
    
    def _preprocess(self):
        self.df['ingredients_list'] = self.df['ingredients'].apply(parse_list_string)
        self.df['step_list'] = self.df['step'].apply(parse_list_string)
        
        self.df['title_clean'] = self.df['title'].apply(clean_text)
        self.df['description_clean'] = self.df['description'].fillna('').apply(clean_text)
        self.df['step_clean'] = self.df['step_list'].apply(lambda x: ' '.join([clean_text(s) for s in x]))
        self.df['combined_text'] = self.df['title_clean'] + ' ' + self.df['description_clean'] + ' ' + self.df['step_clean']
        
        self.df['cook_time_minutes'] = self.df['cook_time'].apply(parse_cook_time)
        self.df['calories_numeric'] = self.df['calories'].apply(parse_calories)
        
        self.df['ingredients_clean'] = self.df['ingredients_list'].apply(
            lambda x: set([clean_text(ing) for ing in x if ing])
        )
        
        # Ingredient text for TF-IDF
        self.df['ingredients_text'] = self.df['ingredients_list'].apply(
            lambda x: ' '.join([clean_text(ing) for ing in x if ing])
        )
        
        # Extract keywords for keyword-based recommendation
        self.df['keywords'] = self.df['title'].apply(extract_keywords)
        
        print("Data preprocessing completed!")
    
    def _build_similarity_matrices(self):
        print("Building TF-IDF similarity...")
        tfidf_vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), min_df=1, max_df=0.95)
        tfidf_matrix = tfidf_vectorizer.fit_transform(self.df['combined_text'])
        self.tfidf_sim = cosine_similarity(tfidf_matrix)
        
        print("Building Ingredient TF-IDF similarity...")
        ing_tfidf_vectorizer = TfidfVectorizer(max_features=2000, ngram_range=(1, 2), min_df=2, max_df=0.8)
        ing_tfidf_matrix = ing_tfidf_vectorizer.fit_transform(self.df['ingredients_text'])
        self.ing_tfidf_sim = cosine_similarity(ing_tfidf_matrix)
        
        print("Building Keyword similarity...")
        # Reuse Jaccard function but for keywords only
        def jaccard_similarity(set1, set2):
            if len(set1) == 0 and len(set2) == 0:
                return 0.0
            intersection = len(set1.intersection(set2))
            union = len(set1.union(set2))
            return intersection / union if union > 0 else 0.0
        
        def compute_jaccard_similarity_matrix(items_list):
            n = len(items_list)
            similarity_matrix = np.zeros((n, n))
            
            for i in range(n):
                for j in range(i, n):
                    sim = jaccard_similarity(items_list[i], items_list[j])
                    similarity_matrix[i][j] = sim
                    similarity_matrix[j][i] = sim
            
            return similarity_matrix
        
        self.keyword_sim = compute_jaccard_similarity_matrix(self.df['keywords'].tolist())
        
        print("Building Hybrid similarity...")
        self.hybrid_sim = compute_hybrid_similarity(
            self.tfidf_sim, self.ing_tfidf_sim,
            w_tfidf=0.4, w_ing_tfidf=0.6
        )
        
        print("All similarity matrices ready!")
    
    def recommend(self, title, method='hybrid', top_n=5):
        """
        Recommend recipes based on title
        
        Args:
            method: 'tfidf', 'ing_tfidf', 'keyword', 'hybrid'
            top_n: Number of recommendations
        """
        matches = self.df[self.df['title'].str.contains(title, case=False, na=False)]
        if len(matches) == 0:
            print(f"No recipe found with title containing: {title}")
            return None
        
        recipe_idx = matches.index[0]
        print(f"\nInput: {self.df.loc[recipe_idx, 'title']}")
        
        return self.recommend_by_index(recipe_idx, method=method, top_n=top_n)
    
    def recommend_by_index(self, idx, method='hybrid', top_n=5):
        """
        Recommend recipes by index
        
        Args:
            method: 'tfidf', 'ing_tfidf', 'keyword', 'hybrid'
            top_n: Number of recommendations
        """
        if idx < 0 or idx >= len(self.df):
            print(f"Invalid index: {idx}")
            return None
        
        if method == 'tfidf':
            return get_tfidf_recommendations(idx, self.tfidf_sim, self.df, top_n)
        elif method == 'ing_tfidf':
            return get_ingredient_tfidf_recommendations(idx, self.ing_tfidf_sim, self.df, top_n)
        elif method == 'keyword':
            return get_keyword_recommendations(idx, self.keyword_sim, self.df, top_n)
        elif method == 'hybrid':
            return get_hybrid_recommendations(
                idx, self.hybrid_sim, self.tfidf_sim,
                self.ing_tfidf_sim, self.df, top_n
            )
        else:
            print(f"Unknown method: {method}")
            return None

In [22]:
# Khởi tạo recommender instance (4 methods: TF-IDF, Ing TF-IDF, Keyword, Hybrid)
print("Initializing Food Recommender with 4 methods...")
recommender_v3 = FoodRecommender(df)

Initializing Food Recommender with 4 methods...
Data preprocessing completed!
Building TF-IDF similarity...
Building Ingredient TF-IDF similarity...
Building Keyword similarity...
Building Hybrid similarity...
Weights: TF-IDF=0.40, Ingredient TF-IDF=0.60
All similarity matrices ready!


## 6. Testing All Methods - Comprehensive Comparison

In [29]:
# So sánh TẤT CẢ phương pháp với cùng 1 món
test_recipe = "gà kho"

print("="*100)
print("TF-IDF METHOD (Text-based)")
print("="*100)
display(recommender_v3.recommend(test_recipe, method='tfidf', top_n=5))

print("\n" + "="*100)
print("INGREDIENT TF-IDF METHOD (Better Ingredient Matching)")
print("="*100)
display(recommender_v3.recommend(test_recipe, method='ing_tfidf', top_n=5))

print("\n" + "="*100)
print("KEYWORD-BASED METHOD")
print("="*100)
display(recommender_v3.recommend(test_recipe, method='keyword', top_n=5))

print("\n" + "="*100)
print("HYBRID METHOD (TF-IDF + Ingredient TF-IDF)")
print("="*100)
display(recommender_v3.recommend(test_recipe, method='hybrid', top_n=5))

TF-IDF METHOD (Text-based)

Input: Gà kho sả nghệ - món ngon miền Trung mùa đông


,title,type_of_food,calories,cook_time,tfidf_score
197,Gà rang gừng - món ngon mùa đông,Món ngon hàng ngày,2.814 kcal,45 phút,0.572038
428,Hướng dẫn cách làm Gà rang gừng - Món đơn giản...,Món ngon hàng ngày,2.074 kcal,35 phút,0.505393
19,Gà nổ muối hột,Món Tết,2.541 kcal,50 phút,0.460820
67,Khô gà lá chanh nhâm nhi ngày Tết,Món Tết,2.154 kcal,90 phút,0.458636
454,Món cà ri gà - món ấm áp cho ngày lạnh,Món ngon ngày lạnh,2.030 kcal,45 phút,0.451897



INGREDIENT TF-IDF METHOD (Better Ingredient Matching)

Input: Gà kho sả nghệ - món ngon miền Trung mùa đông


,title,type_of_food,calories,cook_time,ing_tfidf_score
105,"Món lòng xào nghệ chuẩn vị, giòn dai, thơm ngo...",Món ngon hàng ngày,1.080 kcal,35 phút,0.503469
197,Gà rang gừng - món ngon mùa đông,Món ngon hàng ngày,2.814 kcal,45 phút,0.485793
124,Ếch xào măng - món chân quê vào nhà hàng,Món ngon hàng ngày,940 kcal,35 phút,0.456271
224,"Canh chua cá khoai mềm, ngọt tự nhiên",Món ngon hàng ngày,1.053 kcal,30 phút,0.441209
359,Rạm kho lá lốt – món ngon Thái Bình,Món ngon hàng ngày,708 kcal,40 phút,0.431739



KEYWORD-BASED METHOD

Input: Gà kho sả nghệ - món ngon miền Trung mùa đông


,title,type_of_food,calories,cook_time,keyword_score
197,Gà rang gừng - món ngon mùa đông,Món ngon hàng ngày,2.814 kcal,45 phút,0.454545
213,Thịt gà nấu đông - món ngon miền Bắc,Món ngon hàng ngày,1.439 kcal,90 phút,0.416667
28,Thịt kho măng khô- món ngon Tết miền Trung,Món Tết,3.043 kcal,75 phút,0.384615
42,Thịt ngâm mắm - món ngon miền Trung,Món Tết,2.835 kcal,40 phút,0.333333
587,Chẻo lạc - món ngon dân dã miền Tây xứ Nghệ,Món ngon theo vùng miền,2.278 kcal,45 phút,0.307692



HYBRID METHOD (TF-IDF + Ingredient TF-IDF)

Input: Gà kho sả nghệ - món ngon miền Trung mùa đông


,title,type_of_food,calories,cook_time,hybrid_score,tfidf_score,ing_tfidf_score
197,Gà rang gừng - món ngon mùa đông,Món ngon hàng ngày,2.814 kcal,45 phút,0.520291,0.572038,0.485793
105,"Món lòng xào nghệ chuẩn vị, giòn dai, thơm ngo...",Món ngon hàng ngày,1.080 kcal,35 phút,0.440849,0.346918,0.503469
19,Gà nổ muối hột,Món Tết,2.541 kcal,50 phút,0.420568,0.460820,0.393733
359,Rạm kho lá lốt – món ngon Thái Bình,Món ngon hàng ngày,708 kcal,40 phút,0.392038,0.332486,0.431739
67,Khô gà lá chanh nhâm nhi ngày Tết,Món Tết,2.154 kcal,90 phút,0.382399,0.458636,0.331574


## 7. Final Summary

### 4 Phương pháp Content-Based Recommendation đã implement:

1. **TF-IDF Based**: Text similarity (title + description + steps) với Cosine similarity
   - Ưu: Tốt cho text matching, phát hiện món ăn có cách nấu tương tự
   - Nhược: Phụ thuộc vào chất lượng text
   - Use case: Tìm món có description/cách nấu giống nhau

2. **Ingredient TF-IDF** **RECOMMENDED**: TF-IDF trên ingredient text
   - Ưu: Xử lý variations trong ingredient naming, gán trọng số cho ingredients
   - Ưu: Không bị exact string matching problem
   - **Best choice cho ingredient-based recommendation**
   - Use case: Tìm món có nguyên liệu tương đồng

3. **Keyword/Tag-Based**: Extract keywords từ title (thịt bò, xào, canh...) và Jaccard similarity
   - Ưu: Đơn giản, focus vào main ingredients và cooking methods
   - Ưu: Dễ hiểu và giải thích, nhanh
   - Use case: Quick matching based on key terms

4. **Hybrid**: Weighted combination - TF-IDF (0.4) + Ingredient TF-IDF (0.6)
   - Ưu: Kết hợp text và ingredient signals, robust
   - Ưu: Cân bằng giữa text similarity và ingredient similarity
   - Use case: Best overall personalized recommendation

### So Sánh và Khuyến Nghị:

| Method | Độ chính xác | Speed | Complexity | Best For |
|--------|--------------|-------|------------|----------|
| TF-IDF | 3/5 | Trung bình | Đơn giản | Text/description matching |
| **Ing TF-IDF** | 4.5/5 | Trung bình | Đơn giản | **Best ingredient matching** |
| Keyword | 3/5 | Nhanh | Đơn giản | Quick keyword matching |
| Hybrid | 4.5/5 | Trung bình | Trung bình | Overall best |

### Top 3 Methods Recommended:
1. **Ingredient TF-IDF** - Best cho ingredient-based recommendation
2. **Hybrid** - Best overall khi kết hợp text + ingredient signals
3. **Keyword** - Nhanh nhất, tốt cho quick search

In [30]:
# Class FoodRecommender hoàn chỉnh - tích hợp tất cả phương pháp
class FoodRecommender:
    
    def __init__(self, df):
        self.df = df.copy()
        self._preprocess()
        self._build_similarity_matrices()
    
    def _preprocess(self):
        self.df['ingredients_list'] = self.df['ingredients'].apply(parse_list_string)
        self.df['step_list'] = self.df['step'].apply(parse_list_string)
        
        self.df['title_clean'] = self.df['title'].apply(clean_text)
        self.df['description_clean'] = self.df['description'].fillna('').apply(clean_text)
        self.df['step_clean'] = self.df['step_list'].apply(lambda x: ' '.join([clean_text(s) for s in x]))
        self.df['combined_text'] = self.df['title_clean'] + ' ' + self.df['description_clean'] + ' ' + self.df['step_clean']
        
        self.df['cook_time_minutes'] = self.df['cook_time'].apply(parse_cook_time)
        self.df['calories_numeric'] = self.df['calories'].apply(parse_calories)
        
        self.df['ingredients_clean'] = self.df['ingredients_list'].apply(
            lambda x: set([clean_text(ing) for ing in x if ing])
        )
        
        # Ingredient text for TF-IDF
        self.df['ingredients_text'] = self.df['ingredients_list'].apply(
            lambda x: ' '.join([clean_text(ing) for ing in x if ing])
        )
        
        # Extract keywords
        self.df['keywords'] = self.df['title'].apply(extract_keywords)
        
        print("Data preprocessing completed!")
    
    def _build_similarity_matrices(self):
        print("Building TF-IDF similarity...")
        tfidf_vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), min_df=1, max_df=0.95)
        tfidf_matrix = tfidf_vectorizer.fit_transform(self.df['combined_text'])
        self.tfidf_sim = cosine_similarity(tfidf_matrix)
        
        print("Building Ingredient TF-IDF similarity...")
        ing_tfidf_vectorizer = TfidfVectorizer(max_features=2000, ngram_range=(1, 2), min_df=2, max_df=0.8)
        ing_tfidf_matrix = ing_tfidf_vectorizer.fit_transform(self.df['ingredients_text'])
        self.ing_tfidf_sim = cosine_similarity(ing_tfidf_matrix)
        
        print("Building Keyword similarity...")
        def jaccard_similarity(set1, set2):
            if len(set1) == 0 and len(set2) == 0:
                return 0.0
            intersection = len(set1.intersection(set2))
            union = len(set1.union(set2))
            return intersection / union if union > 0 else 0.0
        
        def compute_jaccard_similarity_matrix(items_list):
            n = len(items_list)
            similarity_matrix = np.zeros((n, n))
            
            for i in range(n):
                for j in range(i, n):
                    sim = jaccard_similarity(items_list[i], items_list[j])
                    similarity_matrix[i][j] = sim
                    similarity_matrix[j][i] = sim
            
            return similarity_matrix
        
        self.keyword_sim = compute_jaccard_similarity_matrix(self.df['keywords'].tolist())
        
        print("Building Hybrid similarity...")
        self.hybrid_sim = compute_hybrid_similarity(
            self.tfidf_sim, self.ing_tfidf_sim,
            w_tfidf=0.4, w_ing_tfidf=0.6
        )
        
        print("All similarity matrices ready!")
    
    def recommend(self, title, method='hybrid', top_n=5):
        matches = self.df[self.df['title'].str.contains(title, case=False, na=False)]
        if len(matches) == 0:
            print(f"No recipe found with title containing: {title}")
            return None
        
        recipe_idx = matches.index[0]
        print(f"\nInput: {self.df.loc[recipe_idx, 'title']}")
        
        if method == 'tfidf':
            return get_tfidf_recommendations(recipe_idx, self.tfidf_sim, self.df, top_n)
        elif method == 'ing_tfidf':
            return get_ingredient_tfidf_recommendations(recipe_idx, self.ing_tfidf_sim, self.df, top_n)
        elif method == 'keyword':
            return get_keyword_recommendations(recipe_idx, self.keyword_sim, self.df, top_n)
        else:
            return get_hybrid_recommendations(
                recipe_idx, self.hybrid_sim, self.tfidf_sim, 
                self.ing_tfidf_sim, self.df, top_n
            )
    
    def recommend_by_index(self, idx, method='hybrid', top_n=5):
        if idx < 0 or idx >= len(self.df):
            print(f"Invalid index: {idx}")
            return None
        
        print(f"\nInput: {self.df.iloc[idx]['title']}")
        
        if method == 'tfidf':
            return get_tfidf_recommendations(idx, self.tfidf_sim, self.df, top_n)
        elif method == 'ing_tfidf':
            return get_ingredient_tfidf_recommendations(idx, self.ing_tfidf_sim, self.df, top_n)
        elif method == 'keyword':
            return get_keyword_recommendations(idx, self.keyword_sim, self.df, top_n)
        else:
            return get_hybrid_recommendations(
                idx, self.hybrid_sim, self.tfidf_sim,
                self.ing_tfidf_sim, self.df, top_n
            )

In [31]:
# Khởi tạo recommender instance
print("Initializing Food Recommender with 4 methods (TF-IDF, Ing TF-IDF, Keyword, Hybrid)...")
recommender = FoodRecommender(df)

Initializing Food Recommender with 4 methods (TF-IDF, Ing TF-IDF, Keyword, Hybrid)...
Data preprocessing completed!
Building TF-IDF similarity...
Building Ingredient TF-IDF similarity...
Building Keyword similarity...
Building Hybrid similarity...
Weights: TF-IDF=0.40, Ingredient TF-IDF=0.60
All similarity matrices ready!


## 8. Export Functions for Evaluation

In [32]:
# Các hàm export dữ liệu cho team Evaluation (4 methods: TF-IDF, Ing TF-IDF, Keyword, Hybrid)
def export_recommendations_for_evaluation(recommender, sample_indices, 
                                         methods=['tfidf', 'ing_tfidf', 'keyword', 'hybrid'], 
                                         top_n=10):
    results = {method: [] for method in methods}
    
    for idx in sample_indices:
        recipe_title = recommender.df.iloc[idx]['title']
        
        for method in methods:
            recs = recommender.recommend_by_index(idx, method=method, top_n=top_n)
            if recs is not None:
                # Get score column name
                if method == 'ing_tfidf':
                    score_col = 'ing_tfidf_score'
                elif method == 'hybrid':
                    score_col = 'hybrid_score'
                else:
                    score_col = f'{method}_score'
                
                rec_data = {
                    'query_idx': idx,
                    'query_title': recipe_title,
                    'recommendations': recs['title'].tolist(),
                    'scores': recs[score_col].tolist()
                }
                results[method].append(rec_data)
    
    return results

def get_all_similarity_scores(recommender, query_idx, top_n=None):
    n_recipes = len(recommender.df)
    
    results = pd.DataFrame({
        'idx': range(n_recipes),
        'title': recommender.df['title'].values,
        'type_of_food': recommender.df['type_of_food'].values,
        'tfidf_score': recommender.tfidf_sim[query_idx],
        'ing_tfidf_score': recommender.ing_tfidf_sim[query_idx],
        'keyword_score': recommender.keyword_sim[query_idx],
        'hybrid_score': recommender.hybrid_sim[query_idx]
    })
    
    results = results[results['idx'] != query_idx]
    results = results.sort_values('hybrid_score', ascending=False)
    
    if top_n:
        results = results.head(top_n)
    
    return results.reset_index(drop=True)

def export_to_csv(recommender, output_dir='../evaluation_data'):
    import os
    os.makedirs(output_dir, exist_ok=True)
    
    recipe_data = recommender.df[['title', 'type_of_food', 'calories', 'cook_time', 'source']].copy()
    recipe_data.to_csv(f'{output_dir}/recipes_info.csv', index=True, encoding='utf-8-sig')
    
    np.save(f'{output_dir}/tfidf_similarity.npy', recommender.tfidf_sim)
    np.save(f'{output_dir}/ing_tfidf_similarity.npy', recommender.ing_tfidf_sim)
    np.save(f'{output_dir}/keyword_similarity.npy', recommender.keyword_sim)
    np.save(f'{output_dir}/hybrid_similarity.npy', recommender.hybrid_sim)
    
    print(f"Exported all data to: {output_dir}")

In [33]:
# Test với recipe có nguyên liệu phổ biến hơn
print("Testing with recipe 100 (has common ingredients):")
scores_df = get_all_similarity_scores(recommender_v3, query_idx=100, top_n=10)
display(scores_df)

print("\n" + "="*80)
print("Testing with recipe 0 (unique ingredients - expect low Jaccard):")
scores_df_0 = get_all_similarity_scores(recommender_v3, query_idx=0, top_n=10)
display(scores_df_0)

Testing with recipe 100 (has common ingredients):


,idx,title,type_of_food,tfidf_score,ing_tfidf_score,keyword_score,hybrid_score
0,168,"Tôm rang thịt ba chỉ măn ngọt, béo ngậy ngon cơm",Món ngon hàng ngày,0.652985,0.799229,0.166667,0.740732
1,712,Tép đồng rang khế - món ngon cho ngày hè nóng nực,Thực đơn cho ngày nắng nóng,0.417406,0.563020,0.250000,0.504775
2,144,Thịt ba chỉ kho trứng cút đậm đà,Món ngon hàng ngày,0.315831,0.615564,0.111111,0.495671
3,184,Cơm cháy kho quẹt giòn rụm ăn kèm rau củ quả,Món ngon hàng ngày,0.365733,0.527847,0.000000,0.463001
4,123,"Thịt ba chỉ chiên mắm da giòn, thịt mềm",Món ngon hàng ngày,0.298762,0.562674,0.111111,0.457109
5,28,Thịt kho măng khô- món ngon Tết miền Trung,Món Tết,0.236444,0.590257,0.000000,0.448732
6,136,Giả cầy An Phú - món ngon đặc sản Thái Bình,Món ngon hàng ngày,0.329863,0.525311,0.000000,0.447132
7,177,Ba chỉ rang cháy cạnh thơm ngon như ngoài quán,Món ngon hàng ngày,0.459258,0.427693,0.181818,0.440319
8,118,"Món cà pháo ram thịt ba chỉ, lá lốt",Món ngon hàng ngày,0.390269,0.472954,0.100000,0.439880
9,359,Rạm kho lá lốt – món ngon Thái Bình,Món ngon hàng ngày,0.211106,0.551031,0.000000,0.415061



Testing with recipe 0 (unique ingredients - expect low Jaccard):


,idx,title,type_of_food,tfidf_score,ing_tfidf_score,keyword_score,hybrid_score
0,52,"Cách muối hành trắng giòn, để được lâu",Món Tết,0.607566,0.498747,0.363636,0.542275
1,9321,"Cách muối dưa hành giòn ngon, không bị hăng đơ...",Ngày lễ Tết,0.432073,0.260823,0.333333,0.329323
2,23,Gà bóp hành răm kiểu miền Trung,Món Tết,0.282705,0.291654,0.076923,0.288075
3,64,Mứt cà rốt không cần nước vôi trong,Món Tết,0.242179,0.317418,0.000000,0.287322
4,619,Dưa góp kiểu miền Trung,Quà - Món ăn vặt,0.319987,0.234161,0.090909,0.268491
5,335,Cá nục kho keo - món 'hao cơm' ngày mưa,Món ngon hàng ngày,0.164744,0.334325,0.062500,0.266493
6,10057,Cách muối dưa củ cải giòn ngon không hăng bằng...,Món khô - mắm,0.355507,0.196510,0.222222,0.260109
7,7658,Cải chíp luộc xanh giòn bày đĩa đẹp mắt bằng n...,Món hấp,0.128155,0.340214,0.000000,0.255391
8,10042,"Dưa giá hẹ bằng nước vo gạo ngon, nhanh, đơn ...",Món khô - mắm,0.225267,0.265797,0.066667,0.249585
9,400,Dọc mùng muối chua xứ Nghệ,Món ngon hàng ngày,0.180462,0.294730,0.090909,0.249023


In [28]:
# export_to_csv(recommender_v3, output_dir='../evaluation_data')